In [ ]:
%pip install pandas openpyxl
%pip install dash plotly pandas
%pip install pandas dash plotly geopy openpyxl
%pip install requests

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Hannah

# Load data
df = pd.read_excel('FinalSheet.xlsx', engine='openpyxl')

# Strip whitespace and handle basic text cleaning
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# Convert dates to datetime objects
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Clean numeric columns (removing commas, etc.)
df['Grant Amount'] = df['Grant Amount'].astype(str).str.replace(',', '')
df['Grant Amount'] = pd.to_numeric(df['Grant Amount'], errors='coerce')

# Replace placeholders with proper Null values
df.replace(['N/A', 'nan', 'None'], pd.NA, inplace=True)

# Show the first 5 rows
df.head()

,Year,Date,Grant Amount,Project Category 1,Project Category 2,Location/Municipality
0,2015,2015-03-04,4000.0,Education/Competition,NaN,North Shore City; Pitt
1,2015,2015-05-06,50000.0,Tree Planting,NaN,"Lawrenceville, Chateau, and Neville Island"
2,2015,2015-05-06,225000.0,Education/Competition,NaN,Pittsburgh
3,2015,2015-07-08,337600.0,Public Health/Medicine,NaN,Northgate School District *Bellevue/Avalon
4,2015,2015-09-02,400000.0,Emission Reduction,Diesel Reduction/Electrification,Allegheny County


In [3]:
# Spending by Year

# Group by Year and sum the Grant Amount
df_yearly = df.groupby('Year')['Grant Amount'].sum().reset_index()

# Check the transformed data
# print(df_yearly.head())

# Create the Line Chart
fig = px.line(df_yearly, 
            x='Year', 
            y='Grant Amount', 
            title='Annual Spending Trend (2015-2024)',
            markers=True, # Adds dots to the line for easier hovering
            template='plotly_white')

# Enhance the Interactivity & Look
fig.update_traces(line_color="#4d5cfd", line_width=3) # Custom line color
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Spending ($)",
    hovermode="x unified", # Shows all data for that year in one tooltip
    yaxis=dict(
        title="Grant Amount",
        tickprefix="$",    # Adds $ to the start of every number
        tickformat=",",    # Adds commas (e.g., 1,000,000)
    )
)

fig.show()

# Export for Wix
html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
# print(html_snippet)

In [13]:
# TRANSFORM DATA: Handle both Project Category 1 and 2
# We create two dataframes and stack them to ensure "Double Counting"
df1 = df[['Year', 'Grant Amount', 'Project Category 1']].rename(columns={'Project Category 1': 'Project Category'})
df2 = df[['Year', 'Grant Amount', 'Project Category 2']].rename(columns={'Project Category 2': 'Project Category'})

# Combine them and remove rows where the category is empty (mostly from Category 2)
df_combined = pd.concat([df1, df2]).dropna(subset=['Project Category'])

#GROUP DATA: Sum spending by Year and Category
df_grouped = df_combined.groupby(['Year', 'Project Category'])['Grant Amount'].sum().reset_index()

# Get the unique list of categories for the loop and dropdown
categories = sorted(df_grouped['Project Category'].unique())

fig = go.Figure()

# Add a line for each category
for cat in categories:
    temp_df = df_grouped[df_grouped['Project Category'] == cat]
    fig.add_trace(
        go.Scatter(
            x=temp_df['Year'], 
            y=temp_df['Grant Amount'], 
            name=cat, 
            mode='lines+markers', 
            visible=True
        )
    )

# --- CREATE THE DROPDOWN & LAYOUT ---
fig.update_layout(
    updatemenus=[
        {
            "buttons": [
                {
                    "label": "Show All",
                    "method": "update",
                    "args": [{"visible": [True] * len(categories)}, {"title": "All Project Spending"}]
                },
                *[
                    {
                        "label": cat,
                        "method": "update",
                        "args": [
                            {"visible": [c == cat for c in categories]},
                            {"title": f"Spending: {cat}"}
                        ]
                    } for cat in categories
                ]
            ],
            "direction": "down",
            "showactive": True,
            "x": 2.0,           # 1.0 is the far right edge of the chart area
            "xanchor": "right", # Aligns the right side of the menu to the x-coordinate
            "y": 1.2            # Keeps it above the plot
        }
    ],
    width=1000,
    height=600,
    autosize=False,
    margin=dict(l=50, r=50, t=100, b=50),
    xaxis=dict(title="Year", tickmode='linear'), 
    yaxis=dict(
        title="Grant Amount",
        tickprefix="$",    
        tickformat=",",    
    ),
    title={
            'text': "Annual Spending by Project Category (2015-2024)",
            'y': 0.95,           
            'x': 0.5,            
            'xanchor': 'right', 
            'yanchor': 'top'
    },
    hovermode="x unified"
)

fig.show()

# Export for Wix (or any web embedding)
html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
# print(html_snippet) 

In [ ]:
# 2. TRANSFORM DATA: Handle both Project Category 1 and 2
df1 = df[['Year', 'Grant Amount', 'Project Category 1']].rename(columns={'Project Category 1': 'Project Category'})
df2 = df[['Year', 'Grant Amount', 'Project Category 2']].rename(columns={'Project Category 2': 'Project Category'})

# Combine and remove rows where the category is empty
df_combined = pd.concat([df1, df2]).dropna(subset=['Project Category'])

# 3. GROUP DATA: Sum spending by Year and Category
df_grouped = df_combined.groupby(['Year', 'Project Category'])['Grant Amount'].sum().reset_index()

# Get the list of unique categories
categories = sorted(df_grouped['Project Category'].unique())

# --- CREATE THE FIGURE ---
fig = go.Figure()

# Add a trace for each category (this makes each one toggleable)
for cat in categories:
    temp_df = df_grouped[df_grouped['Project Category'] == cat]
    fig.add_trace(
        go.Scatter(
            x=temp_df['Year'], 
            y=temp_df['Grant Amount'], 
            name=cat, 
            mode='lines+markers',
            visible=True, # All visible by default
            hovertemplate="<b>" + cat + "</b><br>Amount: $%{y:,.0f}<extra></extra>"
        )
    )

# --- CONFIGURE LAYOUT FOR COMPARISON ---
fig.update_layout(
    width=1100, # Widened slightly to make room for the legend
    height=600,
    autosize=False,
    margin=dict(l=50, r=250, t=100, b=50), # Large right margin for the legend "menu"
    xaxis=dict(title="Year", tickmode='linear'), 
    yaxis=dict(
        title="Grant Amount",
        tickprefix="$",    
        tickformat=",",    
    ),
    title={
            'text': "Comparative Annual Spending by Project Category (2015-2024)",
            'y': 0.95, 
            'x': 0.5, 
            'xanchor': 'right', 
            'yanchor': 'top'
    },
    # Legend settings to make it act like a checkbox menu
    legend=dict(
        title="<b>Select Categories to Compare:</b><br><i>(Click to toggle on/off)</i>",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02, # Position it to the right of the plot
        itemclick="toggle", # Toggles visibility like a checkbox
        itemdoubleclick="toggleothers", # Double-click to see ONLY that category
        font=dict(size=11),
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="lightgrey",
        borderwidth=1
    ),
    template="plotly_white",
    hovermode="x unified" # Shows all active categories in one hover window for comparison
)

fig.show()

# Export for Wix (Copy the snippet below if needed)
# html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
# print(html_snippet)

In [33]:
# TRANSFORM DATA: Handle both Category 1 and Category 2
# We stack them to ensure a project with two categories appears in both
df1 = df[['Year', 'Grant Amount', 'Project Category 1']].rename(columns={'Project Category 1': 'Category'})
df2 = df[['Year', 'Grant Amount', 'Project Category 2']].rename(columns={'Project Category 2': 'Category'})
df_combined = pd.concat([df1, df2]).dropna(subset=['Category'])

# GROUP DATA: Sum spending by Year and Category
df_grouped = df_combined.groupby(['Year', 'Category'])['Grant Amount'].sum().reset_index()

# Get the list of all years and categories
all_years = sorted(df_grouped['Year'].unique())
all_cats = sorted(df_grouped['Category'].unique())

colors_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Safe
category_colors = {cat: colors_palette[i % len(colors_palette)] for i, cat in enumerate(all_cats)}

# CREATE THE FIGURE
fig = go.Figure()

# Add a separate Bar trace for each year
for year in all_years:
    # Filter for the year
    year_data = df_grouped[df_grouped['Year'] == year].copy()
    
    # --- KEY CHANGE: Remove categories with 0 or No spending in this specific year ---
    year_data = year_data[year_data['Grant Amount'] > 0]
    
    # Sort by amount descending to make the bar chart look neat
    year_data = year_data.sort_values(by='Grant Amount', ascending=False)

    # Generate colors based on the categories actually present this year
    current_bar_colors = [category_colors[cat] for cat in year_data['Category']]
    fig.add_trace(
        go.Bar(
            x=year_data['Category'],
            y=year_data['Grant Amount'],
            name=str(year),
            visible=(year == all_years[0]), # Show first year by default
            marker_color=current_bar_colors, # Assign the list of colors
            hovertemplate="Category: %{x}<br>Total: %{y:$,.0f}<extra></extra>"
        )
    )
# CREATE DROPDOWN BUTTONS
dropdown_buttons = []

for i, year in enumerate(all_years):
    # Only show the trace corresponding to the selected year
    visibility = [False] * len(all_years)
    visibility[i] = True

    dropdown_buttons.append(dict(
        label=f"Year: {year}",
        method="update",
        args=[{"visible": visibility},
            {"title": f"Spending by Project Category in {year}"}]
    ))

# UPDATE LAYOUT
fig.update_layout(
updatemenus=[dict(
        active=0,
        buttons=dropdown_buttons,
        direction="down",
        x=1.1,
        xanchor="left",
        y=1.2,
        yanchor="top",
        showactive=True
    )],
    xaxis=dict(title="Project Category", tickangle=45),
    yaxis=dict(title="Total Grant Amount ($)", tickprefix="$", tickformat=","),
    title={
        'text': f"Spending by Project Category in {all_years[0]}",
        'x': 0.05,
        'xanchor': 'left',
        'y': 0.95
        },
    template="plotly_white",
    height=600,
    width=1000,
    margin=dict(t=120) # Space for the dropdown and title
)

fig.show()

# Export for Wix/Web embedding
# html_snippet = fig.to_html(full_html=False, include_plotlyjs='cdn')
# print(html_snippet)

In [ ]:
# Abby
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Load Excel file
df = pd.read_excel('CAF_DS.xlsx')

# Clean Grant Amount
df['Grant Amount'] = (
    df['Grant Amount']
    .astype(str)
    .str.replace(r'[\$,\s]', '', regex=True)
    .astype(float)
)

# Clean category column
df['Project Category 1'] = df['Project Category 1'].str.strip()

# Aggregate data
cat_spend = (
    df.groupby('Project Category 1')['Grant Amount']
    .sum()
    .reset_index()
    .sort_values('Grant Amount', ascending=False)
)

# --- PIE CHART ---
fig = px.pie(
    cat_spend,
    names='Project Category 1',
    values='Grant Amount',
    title='Grant Spending Distribution by Category',
    hole=0.4  # donut style
)

fig.update_traces(
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>$%{value:,.0f}<br>%{percent}'
)

fig.update_layout(height=500)

fig.show()

In [ ]:
import sys
!{sys.executable} -m pip install plotly

In [ ]:
# Neha

import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# Load data
#df = pd.read_csv('1740CAF(DataSet1).csv', encoding='latin-1')
df = pd.read_excel("CAF_DS.xlsx", sheet_name="DataSet1")

# Clean grant amount column
df["Grant Amount"] = df["Grant Amount"].replace(r'[\$,]', '', regex=True).astype(float)

# Replace N/A with NaN
df["Project Category 2"] = df["Project Category 2"].replace("N/A", pd.NA)

# Get list of all categories
categories = pd.unique(
    pd.concat([df["Project Category 1"], df["Project Category 2"]]).dropna()
)

app = Dash(__name__)

app.layout = html.Div([
    
    html.H2("Grant Funding by Year"),

    dcc.Checklist(
        id='category_selector',
        options=[{"label": c, "value": c} for c in categories] +
                [{"label": "All Categories", "value": "ALL"}],
        value=["ALL"],
        inline=False
    ),

    dcc.Graph(id='grant_graph')
])


@app.callback(
    Output("grant_graph", "figure"),
    Input("category_selector", "value")
)

def update_graph(selected_categories):

    if "ALL" in selected_categories:
        filtered = df.copy()
    else:
        filtered = df[
            df["Project Category 1"].isin(selected_categories) |
            df["Project Category 2"].isin(selected_categories)
        ]

    # Remove duplicate rows so grant amount isn't counted twice
    filtered = filtered.drop_duplicates()

    # Sum grant money by year
    yearly = filtered.groupby("Year")["Grant Amount"].sum().reset_index()

    fig = px.line(
        yearly,
        x="Year",
        y="Grant Amount",
        markers=True,
        title="Grant Money Allocated Per Year"
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
# Neha
import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
from geopy.geocoders import Nominatim
import time

# Load data
df = pd.read_excel("1740CAF(withSheet1).xlsx", sheet_name="Sheet1")

# Clean columns
df.columns = df.columns.str.strip()

# Clean grant amount
df["Grant Amount"] = df["Grant Amount"].replace(r'[\$,]', '', regex=True).astype(float)

# Clean categories
df["Project Category 1"] = df["Project Category 1"].str.strip()
df["Project Category 2"] = df["Project Category 2"].str.strip()

# Combine categories for filtering
df["All Categories"] = df[["Project Category 1", "Project Category 2"]].apply(
    lambda x: [c for c in x if pd.notna(c)], axis=1
)

# ------------------------
# 🌍 GEOCODING (convert location to lat/lon)
# ------------------------
geolocator = Nominatim(user_agent="grant_map")

def get_lat_lon(location):
    try:
        loc = geolocator.geocode(location + ", Pennsylvania")
        if loc:
            return pd.Series([loc.latitude, loc.longitude])
    except:
        pass
    return pd.Series([None, None])

# Only run once (slow!)
df[["lat", "lon"]] = df["Location/Municipality"].dropna().apply(get_lat_lon)

# ------------------------
# Get unique filters
# ------------------------
categories = sorted(set(
    c for sublist in df["All Categories"] for c in sublist
))

years = sorted(df["Year"].dropna().unique())

# ------------------------
# DASH APP
# ------------------------
app = Dash(__name__)

app.layout = html.Div([

    html.H2("Grant Allocation Map"),

    # Category filter
    dcc.Checklist(
        id="category_filter",
        options=[{"label": c, "value": c} for c in categories],
        value=categories
    ),

    # Year filter
    dcc.Dropdown(
        id="year_filter",
        options=[{"label": y, "value": y} for y in years],
        value=years,
        multi=True
    ),

    dcc.Graph(id="map"),

    html.Div(id="details")
])


@app.callback(
    Output("map", "figure"),
    Input("category_filter", "value"),
    Input("year_filter", "value")
)
def update_map(selected_categories, selected_years):

    filtered = df[
        df["Year"].isin(selected_years)
    ]

    # Filter categories
    filtered = filtered[
        filtered["All Categories"].apply(
            lambda cats: any(c in selected_categories for c in cats)
        )
    ]

    # Remove missing locations
    filtered = filtered.dropna(subset=["lat", "lon"])

    fig = px.scatter_mapbox(
        filtered,
        lat="lat",
        lon="lon",
        size="Grant Amount",
        hover_name="Location/Municipality",
        hover_data={
            "Year": True,
            "Grant Amount": True
        },
        zoom=7,
        height=600
    )

    fig.update_layout(mapbox_style="open-street-map")

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
#df.to_csv("geocoded_data.csv", index=False)

In [ ]:
df = pd.read_csv("geocoded_data.csv")

In [ ]:
# Neha 
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dash import Dash, dcc, html, Input, Output

# ─────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────
df = pd.read_excel("1740CAF(withSheet1).xlsx", sheet_name="Sheet1")

df.columns = df.columns.str.strip()

df["Grant Amount"] = (
    df["Grant Amount"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

df["Project Category 1"] = df["Project Category 1"].fillna("").str.strip()
df["Project Category 2"] = df["Project Category 2"].fillna("").str.strip()

# Combine both category columns into a list (ignoring blanks / N/A)
df["All Categories"] = df[["Project Category 1", "Project Category 2"]].apply(
    lambda row: [c for c in row if c and c.upper() != "N/A"], axis=1
)

# ─────────────────────────────────────────
# GEOCODING  (run once; slow on first load)
# ─────────────────────────────────────────
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="grant_map_app")

def get_lat_lon(location):
    if not location or pd.isna(location):
        return pd.Series([None, None])
    try:
        loc = geolocator.geocode(str(location) + ", Pennsylvania")
        if loc:
            return pd.Series([loc.latitude, loc.longitude])
    except Exception:
        pass
    return pd.Series([None, None])

# Only geocode rows that have a location value
mask = df["Location/Municipality"].notna() & (df["Location/Municipality"].str.strip() != "")
df.loc[mask, ["lat", "lon"]] = (
    df.loc[mask, "Location/Municipality"]
    .apply(get_lat_lon)
    .values
)

# ─────────────────────────────────────────
# COUNTY GEOJSON  (Pennsylvania counties)
# ─────────────────────────────────────────
COUNTY_GEOJSON_URL = (
    "https://raw.githubusercontent.com/plotly/datasets/master/"
    "geojson-counties-fips.json"
)

def load_pa_county_geojson():
    r = requests.get(COUNTY_GEOJSON_URL, timeout=15)
    data = r.json()
    pa_features = [f for f in data["features"] if f["id"].startswith("42")]
    return {"type": "FeatureCollection", "features": pa_features}

county_geojson = load_pa_county_geojson()

# Map lowercase county name → FIPS code (expand this list as needed)
PA_COUNTY_FIPS = {
    "allegheny county":    "42003",
    "armstrong county":    "42005",
    "beaver county":       "42007",
    "butler county":       "42019",
    "fayette county":      "42051",
    "greene county":       "42059",
    "indiana county":      "42063",
    "lawrence county":     "42073",
    "washington county":   "42125",
    "westmoreland county": "42129",
}

def get_county_outline_trace(location_name):
    """Return a Choroplethmapbox trace outlining a PA county, or None."""
    if not location_name or pd.isna(location_name):
        return None
    key = str(location_name).lower().strip()
    fips = PA_COUNTY_FIPS.get(key)
    if not fips:
        return None
    return go.Choroplethmapbox(
        geojson=county_geojson,
        locations=[fips],
        z=[1],
        colorscale=[
            [0, "rgba(24,95,165,0.12)"],
            [1, "rgba(24,95,165,0.12)"],
        ],
        marker_line_color="#185FA5",
        marker_line_width=2.5,
        showscale=False,
        hoverinfo="skip",
        name=location_name,
    )

# ─────────────────────────────────────────
# FILTER OPTIONS
# ─────────────────────────────────────────
all_categories = sorted(
    set(c for cats in df["All Categories"] for c in cats)
)
all_years = sorted(df["Year"].dropna().unique().tolist())

# ─────────────────────────────────────────
# DASH APP
# ─────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Grant Allocation Map", style={"marginBottom": "16px"}),

    html.Label("Filter by Project Category", style={"fontWeight": "bold"}),
    dcc.Checklist(
        id="category_filter",
        options=[{"label": " " + c, "value": c} for c in all_categories],
        value=all_categories,
        inline=True,
        style={"marginBottom": "12px"},
    ),

    html.Label("Filter by Year", style={"fontWeight": "bold"}),
    dcc.Dropdown(
        id="year_filter",
        options=[{"label": str(int(y)), "value": y} for y in all_years],
        value=all_years,
        multi=True,
        placeholder="Select one or more years…",
        style={"marginBottom": "16px"},
    ),

    dcc.Graph(id="map"),

    html.Div(
        id="details",
        style={
            "marginTop": "16px",
            "padding": "16px",
            "background": "#f8f9fa",
            "borderRadius": "8px",
            "display": "none",          # hidden until a point is clicked
        },
    ),
], style={"maxWidth": "1100px", "margin": "0 auto", "padding": "24px"})


# ─────────────────────────────────────────
# CALLBACK — update map
# ─────────────────────────────────────────
@app.callback(
    Output("map", "figure"),
    Input("category_filter", "value"),
    Input("year_filter", "value"),
)
def update_map(selected_categories, selected_years):
    filtered = df[df["Year"].isin(selected_years)].copy()

    # Keep rows where at least one category matches
    filtered = filtered[
        filtered["All Categories"].apply(
            lambda cats: any(c in selected_categories for c in cats)
        )
    ]

    # Only rows with geocoded coordinates
    mapped = filtered.dropna(subset=["lat", "lon"])

    # ── Scatter dots ──────────────────────────────────────────────
    fig = px.scatter_mapbox(
        mapped,
        lat="lat",
        lon="lon",
        size="Grant Amount",
        size_max=30,
        opacity=0.85,
        color="Project Category 1",
        hover_name="Location/Municipality",
        hover_data={
            "Year": True,
            "Grant Amount": ":$,.0f",
            "Project Category 1": True,
            "Project Category 2": True,
            "lat": False,
            "lon": False,
        },
        zoom=9,
        center={"lat": 40.44, "lon": -79.99},   # Pittsburgh metro
        height=620,
    )

    # ── County border outlines ────────────────────────────────────
    seen_counties = set()
    for location in filtered["Location/Municipality"].dropna().unique():
        key = str(location).lower().strip()
        if key in PA_COUNTY_FIPS and key not in seen_counties:
            trace = get_county_outline_trace(location)
            if trace:
                fig.add_trace(trace)
                seen_counties.add(key)

    # ── Clean, minimal map style ──────────────────────────────────
    fig.update_layout(
        mapbox_style="carto-positron",   # <-- the key change: no road clutter
        margin={"r": 0, "t": 0, "l": 0, "b": 0},
        legend=dict(
            title="Project Category",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ddd",
            borderwidth=1,
        ),
    )

    return fig


# ─────────────────────────────────────────
# CALLBACK — show grant details on click
# ─────────────────────────────────────────
@app.callback(
    Output("details", "children"),
    Output("details", "style"),
    Input("map", "clickData"),
)
def show_details(click_data):
    hidden = {
        "marginTop": "16px",
        "padding": "16px",
        "background": "#f8f9fa",
        "borderRadius": "8px",
        "display": "none",
    }
    visible = {**hidden, "display": "block"}

    if not click_data:
        return "", hidden

    point = click_data["points"][0]
    location = point.get("hovertext", "")

    # Find matching rows in original data
    matches = df[df["Location/Municipality"] == location]
    if matches.empty:
        return f"No data found for {location}.", visible

    rows = []
    for _, row in matches.iterrows():
        cats = ", ".join(row["All Categories"]) if row["All Categories"] else "N/A"
        rows.append(html.Div([
            html.H4(location, style={"marginBottom": "6px"}),
            html.P(f"Year: {int(row['Year']) if pd.notna(row['Year']) else 'Unknown'}"),
            html.P(f"Grant Amount: ${row['Grant Amount']:,.0f}"),
            html.P(f"Categories: {cats}"),
            html.Hr(),
        ]))

    return rows, visible


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
# Lasya
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('CAF_DS.xlsx') 

# Clean Grant Amount: remove $, commas, spaces → float
df['Grant Amount'] = (
    df['Grant Amount']
    .astype(str)
    .str.replace(r'[\$,\s]', '', regex=True)
    .astype(float)
)

# Normalize category columns (strip whitespace)
df['Project Category 1'] = df['Project Category 1'].str.strip()
df['Project Category 2'] = df['Project Category 2'].str.strip()

print(f"Dataset shape: {df.shape}")
print(f"Years covered: {sorted(df['Year'].unique())}")
print(f"Total grant funding: ${df['Grant Amount'].sum():,.0f}")
df.head(10)

cat_spend = (
    df.groupby('Project Category 1')['Grant Amount']
    .sum()
    .reset_index()
    .sort_values('Grant Amount', ascending=False)
)

fig = px.bar(
    cat_spend,
    x='Project Category 1',
    y='Grant Amount',
    title='Total Grant Spending by Project Category',
    color='Grant Amount',
    color_continuous_scale='Teal',
    text='Grant Amount'
)
fig.update_traces(
    texttemplate='$%{text:,.0f}',
    textposition='outside'
)
fig.update_layout(
    xaxis_title='Project Category',
    yaxis_title='Total Grant Amount ($)',
    xaxis_tickangle=-30,
    coloraxis_showscale=False,
    height=500
)
fig.show()

In [ ]:
# Filter for Emission Reduction only
emission_df = df[
    (df['Project Category 1'] == 'Emission Reduction') |
    (df['Project Category 2'] == 'Emission Reduction')
]

# Group by Year
emission_yearly = (
    emission_df.groupby('Year')['Grant Amount']
    .sum()
    .reset_index()
)

# Line chart over time
fig = px.line(
    emission_yearly,
    x='Year',
    y='Grant Amount',
    title='Emission Reduction Grant Spending Over Time',
    markers=True,
    template='plotly_white'
)
fig.update_traces(
    line_color="#2ca02c",
    line_width=3,
    hovertemplate="<b>Year:</b> %{x}<br><b>Spent:</b> $%{y:,.0f}<extra></extra>"
)
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Grant Amount ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    hovermode="x unified"
)
fig.show()